# Observatório de Indicadores: Classificação de ODS em Artigos

**Objetivo:** Este notebook implementa a classificação de artigos científicos com base nos Objetivos de Desenvolvimento Sustentável (ODS).

**Metodologia (Opção 2):**
1.  Carrega o dataset de artigos já coletado (em formato CSV).
2.  Prepara os campos de texto (título, resumo, palavras-chave) para a busca.
3.  Utiliza as listas de termos de busca da Elsevier para cada ODS.
4.  Realiza uma busca textual local para identificar a presença desses termos em cada artigo.
5.  Cria novas colunas no dataset para indicar a quais ODS cada artigo está associado.
6.  Salva o novo dataset enriquecido para análises futuras.

In [1]:
# Instala a biblioteca necessária para a análise de dados.
# O 'pandas' é fundamental para manipulação de tabelas (dataframes).
!pip install pandas

In [8]:
# Célula 3: Código (Importação e Carregamento dos Dados) - USANDO O ARQUIVO LIMPO

import pandas as pd

# --- CONFIGURAÇÃO ---
# Apontando para o novo arquivo de dados processado
NOME_ARQUIVO_ENTRADA = "/workspaces/sismm-cti-amazon/data/processed/scopus_dados_limpos_temp.csv"
NOME_ARQUIVO_SAIDA = "artigos_classificados_ods.csv"

# --- LEITURA DO ARQUIVO ---
try:
    # Usamos o leitor padrão, que é ideal para arquivos CSV limpos (separados por vírgula)
    df = pd.read_csv(NOME_ARQUIVO_ENTRADA)
    
    print(f"Arquivo '{NOME_ARQUIVO_ENTRADA}' carregado com sucesso!")
    print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.")
    
    # --- VERIFICAÇÃO ---
    print("\nColunas identificadas no arquivo:")
    print(df.columns.tolist())

except Exception as e:
    print(f"ERRO CRÍTICO ao ler o arquivo: {e}")

# Exibe as 5 primeiras linhas para verificar os dados e a estrutura
print("\nAmostra dos dados carregados:")
display(df.head())

Arquivo '/workspaces/sismm-cti-amazon/data/processed/scopus_dados_limpos_temp.csv' carregado com sucesso!
O dataset possui 800 linhas e 32 colunas.

Colunas identificadas no arquivo:
['authors', 'author_full_names', 'authors_id', 'title', 'year', 'source_title', 'volume', 'issue', 'page_start', 'page_end', 'page_count', 'cited_by', 'doi', 'link', 'affiliations', 'authors_with_affiliations', 'abstract', 'author_keywords', 'index_keywords', 'funding_details', 'references', 'correspondence_address', 'publisher', 'issn', 'coden', 'language_of_original_document', 'abbreviated_source_title', 'document_type', 'publication_stage', 'open_access', 'source', 'eid']

Amostra dos dados carregados:


,authors,author_full_names,authors_id,title,year,source_title,volume,issue,page_start,page_end,...,publisher,issn,coden,language_of_original_document,abbreviated_source_title,document_type,publication_stage,open_access,source,eid
0,do Nascimento C.S.; de Almeida Cruz I.; do Nas...,"do Nascimento, Cristiano Souza (57212932024); ...",57212932024; 58355365500; 26028309100; 5595411...,Technological properties of wood from small di...,2023,European Journal of Forest Research,142,5,1225,1238,...,Springer Science and Business Media Deutschlan...,16124669,nao_informado,English,Eur. J. For. Res.,Article,Final,nao_informado,Scopus,2-s2.0-85162981044
1,da Silva A.S.O.; de Carvalho J.O.P.; Dionisio ...,"da Silva, Antonia Sandra Oliveira (58119830300...",58119830300; 7102474765; 57189244151; 36928208...,Structure of Eschweilera amazonica R. Knuth (m...,2023,Scientia Forestalis/Forest Sciences,51,0,0,0,...,University of Sao Paolo,14139324,nao_informado,Portuguese,Sci Forest,Article,Final,All Open Access; Gold Open Access,Scopus,2-s2.0-85149013333
2,Daly D.C.,"Daly, Douglas C. (7102740855)",7102740855,"A rare new species of Protium from Rondônia, B...",2023,Brittonia,75,2,210,214,...,Springer,0007196X,BRTAA,English,Brittonia,Article,Final,nao_informado,Scopus,2-s2.0-85160833451
3,Londoño-Echeverri Y.; Trujillo-López A.M.; Pér...,"Londoño-Echeverri, Yeison (57221612640); Truji...",57221612640; 57221607661; 58531148900,A new species of Conchocarpus and first record...,2023,Phytotaxa,601,2,174,184,...,Magnolia Press,11793155,nao_informado,English,Phytotaxa,Article,Final,nao_informado,Scopus,2-s2.0-85167577291
4,Corrêa P.G.; Moura L.G.S.; Amaral A.C.F.; do A...,"Corrêa, Pollyane Gomes (58001294800); Moura, L...",58001294800; 58002168600; 7005934688; 57209362...,Chemical and nutritional characterization of A...,2023,Food Research International,163,0,0,0,...,Elsevier Ltd,09639969,FORIE,English,Food Res. Int.,Article,Final,nao_informado,Scopus,2-s2.0-85143804200


### Passo 1: Preparação dos Dados para a Busca

Para realizar a busca de forma eficiente, vamos:
1.  Juntar as colunas de texto relevantes (`titulo`, `resumo`, `palavras_chave`) em uma única coluna.
2.  Tratar valores ausentes (NaN) para evitar erros.
3.  Converter todo o texto para minúsculas para garantir que a busca não seja sensível a maiúsculas/minúsculas (ex: "Solar" e "solar" serão tratados da mesma forma).

In [10]:
# Célula 5: Código (Limpeza e Concatenação de Texto) - VERSÃO ATUALIZADA

# --- CONFIGURAÇÃO ---
# Nomes exatos das colunas do novo arquivo 'scopus_dados_limpos_temp.csv'
COLUNA_TITULO = 'title'
COLUNA_RESUMO = 'abstract'
COLUNA_PALAVRAS_CHAVE = 'author_keywords'

# --------------------------------------------------------------------

# Substitui valores nulos (NaN) por uma string vazia para evitar erros na concatenação
df[COLUNA_TITULO] = df[COLUNA_TITULO].fillna('')
df[COLUNA_RESUMO] = df[COLUNA_RESUMO].fillna('')
df[COLUNA_PALAVRAS_CHAVE] = df[COLUNA_PALAVRAS_CHAVE].fillna('')

# Cria uma coluna unificada para a busca e a converte para minúsculas
df['texto_busca'] = (df[COLUNA_TITULO] + ' ' + df[COLUNA_RESUMO] + ' ' + df[COLUNA_PALAVRAS_CHAVE]).str.lower()

print("Coluna 'texto_busca' criada com sucesso.")
# Exibe um exemplo do texto combinado
print("\nExemplo de texto combinado:")
print(df['texto_busca'].iloc[0])

Coluna 'texto_busca' criada com sucesso.

Exemplo de texto combinado:
technological properties of wood from small diameter in an area of forest exploitation of reduced impact in the tropical forest the study characterized the technological properties of eight woods from small-diameter forest species with vast occurrence in the central amazon (amazonas/brazil), as a potential for indication in a forest exploration plan. samples were obtained from the managed area, and 24 trees (diameter at breast height ≤ 50 cm) were used in chemical and physical–mechanical determinations. among the eight species studied, the wood matá-matá yellow (eschweilera odora) had the highest concentration of extractives and polyphenols (7.08 and 2.63%), while piãozinho (microdropsis scleroxylon) had the highest lignin content (34.80%). for the physical–mechanical properties, the basic density ranged from 0.56 for ingá-white (inga alba), and 0.93 g/cm3 for piãozinho, and matá-matá black (eschweilera truncata) sho

### Passo 2: Definição dos Termos de Busca para cada ODS

Aqui definimos as "queries" de busca para cada ODS, baseadas nas sugestões da Elsevier.
- Usamos um dicionário onde a chave é o nome do ODS e o valor é uma string com os termos.
- Os termos são separados por `|`, que funciona como um "OU" em expressões regulares (regex).

**Atenção:** Os termos abaixo são apenas um **exemplo**. Você deve substituir pelas listas de termos completas que você possui.

In [11]:
# Célula 7: Código (Dicionário de Queries ODS - Gerado Automaticamente)

import re

# Nomes descritivos para cada ODS, para usar como chave no dicionário
ods_names = {
    1: "ODS 1: Erradicação da Pobreza",
    2: "ODS 2: Fome Zero e Agricultura Sustentável",
    3: "ODS 3: Saúde e Bem-Estar",
    4: "ODS 4: Educação de Qualidade",
    5: "ODS 5: Igualdade de Gênero",
    6: "ODS 6: Água Potável e Saneamento",
    7: "ODS 7: Energia Limpa e Acessível",
    8: "ODS 8: Trabalho Decente e Crescimento Econômico",
    9: "ODS 9: Indústria, Inovação e Infraestrutura",
    10: "ODS 10: Redução das Desigualdades",
    11: "ODS 11: Cidades e Comunidades Sustentáveis",
    12: "ODS 12: Consumo e Produção Responsáveis",
    13: "ODS 13: Ação Contra a Mudança Global do Clima",
    14: "ODS 14: Vida na Água",
    15: "ODS 15: Vida Terrestre",
    16: "ODS 16: Paz, Justiça e Instituições Eficazes"
}

# Cole aqui o texto bruto com as expressões de busca da Scopus
raw_scopus_queries = """
ODS (Meta nº)	Expressão de busca
1	TITLE-ABS-KEY ( ( {extreme poverty} OR {poverty alleviation} OR {poverty eradication} OR {poverty reduction} OR {international poverty line} OR ( {financial aid} AND {poverty} ) OR ( {financial aid} AND {poor} ) OR ( {financial aid} AND {north-south divide} ) OR ( {financial development} AND {poverty} ) OR {financial empowerment} OR {distributional effect} OR {distributional effects} OR {child labor} OR {child labour} OR {development aid} OR {social protection} OR {social protection system} OR ( {social protection} AND access ) OR microfinanc* OR micro-financ* OR {resilience of the poor} OR ( {safety net} AND {poor} OR {vulnerable} ) OR ( {economic resource} AND access ) OR ( {economic resources} AND access ) OR {food bank} OR {food banks} ) )
2	TITLE-ABS-KEY ( ( {land tenure rights} OR ( smallholder AND ( farm OR forestry OR pastoral OR agriculture OR fishery OR {food producer} OR {food producers} ) ) OR malnourish* OR malnutrition OR undernourish* OR {undernutrition} OR {agricultural production} OR {agricultural productivity} OR {agricultural practices} OR {agricultural management} OR {food production} OR {food productivity} OR {food security} OR {food insecurity} OR {land right} OR {land rights} OR {land reform} OR {land reforms} OR {resilient agricultural practices} OR ( agriculture AND potassium ) OR fertili?er OR {food nutrition improvement} OR {hidden hunger} OR {genetically modified food} OR ( gmo AND food ) OR {agroforestry practices} OR {agroforestry management} OR {agricultural innovation} OR ( {food security} AND {genetic diversity} ) OR ( {food market} AND ( restriction OR tariff OR access OR {north south divide} OR {development governance} ) ) OR {food governance} OR {food supply chain} OR {food value chain} OR {food commodity market} AND NOT {disease} ) )
3	TITLE-ABS-KEY ( ( ( human AND ( health* OR disease* OR illness* OR medicine* OR mortality ) ) OR {battered child syndrome} OR {cardiovascular disease} OR {cardiovascular diseases} OR {chagas} OR {child abuse} OR {child neglect} OR {child well-being index} OR {youth well-being index} OR {child wellbeing index} OR {youth wellbeing index} OR {water-borne disease} OR {water-borne diseases} OR {water borne disease} OR {water borne diseases} OR {tropical disease} OR {tropical diseases} OR {chronic respiratory disease} OR {chronic respiratory diseases} OR {infectious disease} OR {infectious diseases} OR {sexually-transmitted disease} OR {sexually transmitted disease} OR {sexually-transmitted diseases} OR {sexually transmitted diseases} OR {communicable disease} OR {communicable diseases} OR aids OR hiv OR {human immunodeficiency virus} OR tuberculosis OR malaria OR hepatitis OR polio* OR vaccin* OR cancer* OR diabet* OR {maternal mortality} OR {child mortality} OR {childbirth complications} OR {neonatal mortality} OR {neo-natal mortality} OR {premature mortality} OR {infant mortality} OR {quality adjusted life year} OR {maternal health} OR {preventable death} OR {preventable deaths} OR {tobacco control} OR {substance abuse} OR {drug abuse} OR {tobacco use} OR {alcohol use} OR {substance addiction} OR {drug addiction} OR {tobacco addiction} OR alcoholism OR suicid* OR {postnatal depression} OR {post-natal depression} OR {zika virus} OR dengue OR schistosomiasis OR {sleeping sickness} OR ebola OR {mental health} OR {mental disorder} OR {mental illness} OR {mental illnesses} OR {measles} OR {neglected disease} OR {neglected diseases} OR diarrhea OR diarrhoea OR cholera OR dysentery OR {typhoid fever} OR {traffic accident} OR {traffic accidents} OR {healthy lifestyle} OR {life expectancy} OR {life expectancies} OR {health policy} OR ( {health system} AND ( access OR accessible ) ) OR {health risk} OR {health risks} OR {inclusive health} OR obesity OR {social determinants of health} OR {psychological harm} OR {psychological wellbeing} OR {psychological well-being} OR {psychological well being} OR {public health} ) )
4	TITLE-ABS-KEY ( ( school OR education OR educational ) AND ( {school attendance} OR {school enrollment} OR {school enrolment} OR {inclusive education} OR {educational inequality} OR {education quality} OR {educational enrolment} OR {educational enrollment} OR {adult literacy} OR {numeracy rate} OR {educational environment} OR {educational access} OR ( {development aid} AND {teacher training} ) OR {early childhood education} OR {basic education} OR {affordable education} OR {educational financial aid} OR {school safety} OR {safety in school} OR ( {learning opportunities} AND ( {gender disparities} OR empowerment ) ) OR ( {learning opportunity} AND ( {gender disparities} OR empowerment ) ) OR {youth empowerment} OR {women empowerment} OR {equal opportunities} OR {child labour} OR {child labor} OR {discriminatory} OR {educational inequality} OR {educational gap} OR ( {poverty trap} AND {schooling} ) OR {special education needs} OR {inclusive education system} OR ( {schooling} AND ( {gender disparities} OR {ethnic disparities} OR {racial disparities} ) ) OR {education exclusion} OR {education dropouts} OR {global citizenship} OR {sustainable development education} OR {environmental education} OR {education policy} OR {educational policies} OR {international education} OR {education reform} OR ( {educational reform} AND {developing countries} ) OR {educational governance} OR ( {developing countries} AND {school effects} ) OR {education expenditure} OR {foreign aid} OR ( {teacher training} AND {developing countries} ) OR {teacher attrition} ) AND NOT {health literacy} )
5	TITLE-ABS-KEY ( ( {gender inequality} OR {gender equality} OR {employment equity} OR {gender wage gap} OR {female labor force participation} OR {female labour force participation} OR {women labor force participation} OR {women labour force participation} OR {womens' employment} OR {female employment} OR {women's unemployment} OR {female unemployment} OR ( access AND {family planning services} ) OR {forced marriage} OR {child marriage} OR {forced marriages} OR {child marriages} OR {occupational segregation} OR {women's empowerment} OR {girls' empowerment} OR {female empowerment} OR {female genital mutilation} OR {female genital cutting} OR {domestic violence} OR {women AND violence} OR {girl* AND violence} OR {sexual violence} OR ( {unpaid work} AND {gender inequality} ) OR ( {unpaid care work} AND {gender inequality} ) OR {women's political participation} OR {female political participation} OR {female managers} OR {women in leadership} OR {female leadership} OR {intra-household allocation} OR ( access AND {reproductive healthcare} ) OR {honour killing} OR {honor killing} OR {honour killings} OR {honor killings} OR {antiwomen} OR {anti-women} OR {feminism} OR {misogyny} OR {female infanticide} OR {female infanticides} OR {human trafficking} OR {forced prostitution} OR ( equality AND ( {sexual rights} OR {reproductive rights} OR {divorce rights} ) ) OR {women's rights} OR {gender injustice} OR {gender injustices} OR {gender discrimination} OR {gender disparities} OR {gender gap} OR {female exploitation} OR {household equity} OR {female political participation} OR {women's underrepresentation} OR {female entrepreneurship} OR {female ownership} OR {women's economic development} OR {women's power} OR {gender-responsive budgeting} OR {gender quota} OR ( {foreign aid} AND {women's empowerment} ) OR {gender segregation} OR {gender-based violence} OR {gender participation} OR {female politician} OR {female leader} OR {contraceptive behaviour} OR {women's autonomy} OR {agrarian feminism} OR {microfinance} OR {women's livelihood} OR {women's ownership} OR {female smallholder} OR {gender mainstreaming} ) )
6	TITLE-ABS-KEY ( ( ( ( {Safe} AND ( {water access} OR {drinking water} ) ) OR ( {clean} AND ( {drinking water} OR {water source} ) ) OR ( {water} AND ( {sanitation and hygiene} OR {sanitation & hygiene} OR {quality} OR {resource} ) AND ( {water availability} OR {water-use efficiency} OR {water supply} OR {water supplies} OR {clean water} OR {hygienic toilet} OR {hygienic toilets} OR {antifouling membrane} OR {antifouling membranes} OR {anti-fouling membrane} OR {anti-fouling membranes} OR {water management} OR {aquatic toxicology} OR {water toxicology} OR {aquatic ecotoxicology} OR {water ecotoxicology} ) ) OR ( ( {freshwater} OR {fresh water} ) AND ( {water quality} ) AND ( {pollutant} OR {pollution} OR contamina* ) ) OR ( {freshwater} AND ( {water security} OR {water shortage} OR ({waste water} AND “treatment”) OR ({wastewater} AND “treatment”) OR {water conservation} OR {water footprint} OR {water infrastructure} OR {water pollution} OR {water purification} OR {water use} OR {water uses} OR sanit* OR sewer* ) ) OR ( ( {water} AND ( {ecosystem} OR {eco-system} ) AND ( {protection of} OR {endocrine disruptor} OR {endocrine disruptors} ) ) AND NOT {marine} ) OR ({water} AND {water management} AND ({pollution remediation} OR {pollutant removal})) OR (({groundwater} OR {ground water} OR {ground-water}) AND {freshwater}) OR (({water pollution} OR {water pollutant}) AND ({waste water} AND “treatment”) OR ({wastewater} AND “treatment”)) OR {freshwater availability} OR {fresh water availability} OR {water scarcity} OR {open defecation} OR {blue water} OR {green water} OR {grey water} OR {black water} ) ) AND NOT {global burden of disease study} )
7	TITLE-ABS-KEY ( ( {energy efficiency} OR {energy consumption} OR {energy transition} OR {clean energy technology} OR {energy equity} OR {energy justice} OR {energy poverty} OR {energy policy} OR renewable* OR {2000 Watt society} OR {smart micro-grid} OR {smart grid} OR {smart microgrid} OR {smart micro-grids} OR {smart grids} OR {smart microgrids} OR {smart meter} OR {smart meters} OR {affordable electricity} OR {electricity consumption} OR {reliable electricity} OR {clean fuel} OR {clean cooking fuel} OR {fuel poverty} OR energiewende OR {life-cycle assessment} OR {life cycle assessment} OR {life-cycle assessments} OR {life cycle assessments} OR ( {photochemistry} AND {renewable energy} ) OR photovoltaic OR {photocatalytic water splitting} OR {hydrogen production} OR {water splitting} OR {lithium-ion batteries} OR {lithium-ion battery} OR {heat network} OR {district heat} OR {district heating} OR {residential energy consumption} OR {domestic energy consumption} OR {energy security} OR {rural electrification} OR {energy ladder} OR {energy access} OR {energy conservation} OR {low-carbon society} OR {hybrid renewable energy system} OR {hybrid renewable energy systems} OR {fuel switching} OR ( {foreign development aid} AND {renewable energy} ) OR {energy governance} OR ( {official development assistance} AND {electricity} ) OR ( {energy development} AND {developing countries} ) ) AND NOT ( {wireless sensor network} OR {wireless sensor networks} ) )
8	TITLE-ABS-KEY ( ( {economic growth} OR {economic development policy} OR {employment policy} OR {inclusive economic growth} OR {sustainable growth} OR {economic development} OR {economic globalization} OR {economic globalisation} OR {economic productivity} OR {low-carbon economy} OR {inclusive growth} OR microfinanc* OR micro-financ* OR micro-credit* OR microcredit* OR {equal income} OR {equal wages} OR {decent job} OR {decent jobs} OR {quality job} OR {quality jobs} OR {job creation} OR {full employment} OR {employment protection} OR {informal employment} OR {precarious employment} OR {unemployment} OR {precarious job} OR {precarious jobs} OR microenterprise* OR micro-enterprise* OR {small enterprise} OR {medium enterprise} OR {small enterprises} OR {medium enterprises} OR {small entrepreneur} OR {starting entrepreneur} OR {medium entrepreneur} OR {small entrepreneurs} OR {medium entrepreneurs} OR {starting entrepreneurs} OR {social entrepreneurship} OR {safe working environment} OR {labor market institution} OR {labor market institutions} OR {labour market institution} OR {labour market institutions} OR {forced labour} OR {forced labor} OR {child labour} OR {child labor} OR {labour right} OR {labor right} OR {labour rights} OR {labor rights} OR {modern slavery} OR {human trafficking} OR {child soldier} OR {child soldiers} OR {global jobs} OR {living wage} OR {minimum wage} OR {circular economy} OR {inclusive economy} OR {rural economy} OR {Foreign Development Investment} OR {Aid for Trade} OR {trade unions} OR {trade union} OR {working poor} OR {Not in Education, Employment, or Training} OR {carbon offset} OR {carbon offsetting} OR {carbon offsets} OR {offset project} OR {offset projects} OR {economic diversification} OR {material footprint} OR {resource efficiency} OR ( {cradle to cradle} AND {economy} ) OR {economic decoupling} OR {labour market disparities} OR {sustainable tourism} OR {ecotourism} OR {community-based tourism} OR {tourism employment} OR {sustainable tourism policy} OR {financial access} OR {financial inclusion} OR {access to banking} ) AND NOT {health} )
9	TITLE-ABS-KEY ( ( {industrial growth} OR {industrial diversification} OR {infrastructural development} OR {infrastructural investment} OR {infrastructure investment} OR {public infrastructure} OR {resilient infrastructure} OR {transborder infrastructure} OR {public infrastructures} OR {resilient infrastructures} OR {transborder infrastructures} OR ( {industrial emissions} AND mitigation ) OR {industrial waste management} OR {industrial waste treatment} OR {traffic congestion} OR microenterprise* OR micro-enterprise* OR {small enterprise} OR {medium enterprise} OR {small enterprises} OR {medium enterprises} OR {small entrepreneur} OR {medium entrepreneur} OR {small entrepreneurs} OR {medium entrepreneurs} OR {value chain management} OR ( {broadband access} AND {developing countries} ) OR {manufacturing innovation} OR {manufacturing investment} OR {sustainable transportation} OR {accessible transportation} OR {transportation services} OR {inclusive transportation} OR {R&D investment} OR {green product} OR {green products} OR {sustainable manufacturing} OR ( {cradle to cradle} AND industry ) OR {closed loop supply chain} OR ( industrial AND innovation ) OR {process innovation} OR {product innovation} OR {inclusive innovation} ) )
10	TITLE-ABS-KEY ( ( ( equality AND ( economic OR financial OR socio-economic ) ) OR ( inequality AND ( economic OR financial OR socio-economic ) ) OR {economic reform policy} OR {economic reform policies} OR {political inclusion} OR {social protection policy} OR {social protection policies} OR ( immigration AND NOT ( chemistry OR disease OR biodiversity ) ) OR ( emigration AND NOT ( chemistry OR disease OR biodiversity ) ) OR {foreign direct investment} OR {development gap} OR {development gaps} OR {migrant remittance} OR {responsible migration} OR {migration policy} OR {migration policies} OR {north-south divide} OR ( developing AND ( {tariffs} OR {tariff} OR {zero-tariff} OR {duty-free access} ) ) OR {social exclusion} OR {economic marginali?ation} OR {income inequality} OR {discriminatory law*} OR {discriminatory policies} OR {discriminatory policy} OR {economic empowerment} OR {economic transformation} OR ( {global market} AND {empowerment} ) ) )
11	TITLE-ABS-KEY ( ( city OR cities OR {human settlement} OR {human settlements} OR urban OR metropoli* OR town* OR municipal* ) AND ( gentrification OR congestion OR transportation OR {public transport} OR housing OR slum* OR {sendai framework} OR {Disaster Risk Reduction} OR {DRR} OR {smart city} OR {smart cities} OR {resilient building} OR {resilient buildings} OR {sustainable building} OR {sustainable buildings} OR {building design} OR {buildings design} OR urbani?ation OR {zero energy building} OR {zero energy buildings} OR {zero-energy building} OR {zero-energy buildings} OR {basic service} OR {basic services} OR {governance} OR {citizen participation} OR {collaborative planning} OR {participatory planning} OR {inclusiveness} OR {cultural heritage} OR {natural heritage} OR {UNESCO} OR {disaster} OR {ecological footprint} OR {environmental footprint} OR {waste} OR {pollution} OR {pollutant*} OR {waste water} OR {recycling} OR {circular economy} OR {air quality} OR {green space} OR {green spaces} OR {nature inclusive} OR {nature inclusive building} OR {nature inclusive buildings} ) )
12	TITLE-ABS-KEY ( {environmental pollution} OR {hazardous waste} OR {hazardous chemical} OR {hazardous chemicals} OR {toxic chemical} OR {toxic chemicals} OR {chemical pollution} OR {ozone depletion} OR {pesticide pollution} OR {pesticide stress} OR {pesticide reduction} OR {life cycle assessment} OR {life cycle analysis} OR {life cycle analyses} OR {life-cycle analysis} OR {life-cycle analyses} OR {low carbon economy} OR {low-carbon economy} OR {environmental footprint} OR {material footprint} OR {harvest efficiency} OR {solid waste} OR {waste generation} OR {corporate social responsibility} OR {corporate sustainability} OR {consumer behavior} OR {consumer behaviors} OR {consumer behaviour} OR {consumer behaviours} OR {waste recycling} OR {resource recycling} OR {resource reuse} OR {biobased economy} OR {zero waste} OR {sustainability label} OR {sustainability labelling} OR {global resource extraction} OR {material flow accounting} OR {societal metabolism} OR {food spill} OR {resource spill} OR {resource efficiency} OR {sustainable food consumption} OR {green consumption} OR {sustainable supply chain} OR {circular economy} OR {cradle to cradle} OR {sustainable procurement} OR {sustainable tourism} OR {fossil-fuel subsidies} OR {fossil-fuel expenditure} OR {sustainability label} OR {sustainability labelling} OR ( consumption AND ( {resource use} OR spill ) ) OR ( production AND ( {resource use} OR spill ) ) AND NOT ( {wireless sensor network} OR {wireless sensor networks} OR {wireless network} OR {wireless networks} OR {wireless} OR {disease} OR {astrophysics} ) )
13	TITLE-ABS-KEY ( ( {climate action} OR {climate adaptation} OR {climate change} OR {climate capitalism} OR ipcc OR {climate effect} OR {climate equity} OR {climate feedback} OR {climate finance} OR {climate change financing} OR {climate forcing} OR {climate governance} OR {climate impact} OR {climate investment} OR {climate justice} OR {climate mitigation} OR {climate model} OR {climate models} OR {climate modeling} OR {climate modelling} OR {climate policy} OR {climate policies} OR {climate risk} OR {climate risks} OR {climate services} OR {climate service} OR {climate prediction} OR {climate predictions} OR {climate signal} OR {climate signals} OR {climate tipping point} OR {climate variation} OR {climate variations} OR ecoclimatology OR eco-climatology OR {Green Climate Fund} OR {regional climate} OR {regional climates} OR {urban climate} OR {urban climates} OR ( climate AND ( {adaptive management} OR awareness OR bioeconomy OR carbon OR {decision-making} OR {disaster risk reduction} OR {environmental education} OR {sustainable development education} OR {energy conservation} OR emission* OR extreme OR {food chain} OR {food chains} OR framework OR hazard* OR island* OR {land use} OR megacit* OR consumption OR production OR {small island developing states} OR anthropocene OR atmospher* OR {clean development mechanism} OR {glacier retreat} OR warming OR greenhouse OR {ice-ocean interaction} OR {ice-ocean interactions} OR {nitrogen cycle} OR {nitrogen cycles} OR {ocean acidification} OR {radiative forcing} OR {sea ice} OR {sea level} OR {sea levels} OR {thermal expansion} OR unfccc OR ozone ) ) ) AND NOT ( {drug} OR {geomorphology} ) )
14	TITLE-ABS-KEY ( ( marine OR ocean OR oceans OR sea OR seas OR coast* OR mangrove ) AND ( {water cycle} OR {water cycles} OR {biogeochemical cycle} OR {biogeochemical cycles} OR {oceanic circulation model} OR {oceanic circulation models} OR {oceanic circulation modelling} OR {oceanic circulation modeling} OR {ice-ocean} OR eutrophicat* OR marine OR {coral bleach} OR {coral bleaching} OR {coastal management} OR {coastal habitat} OR {coastal habitats} OR {marine debris} OR {ocean acidification} OR ( acidification AND seawater ) OR {fishery} OR {fisheries} OR {overfishing} OR {sustainable yield} OR {marine protected area} OR {marine protected areas} OR {marine conservation} OR {ecotourism} OR {community based conservation} OR {community-based conservation} OR {marine land slide} OR {marine pollution} OR {nutrient runoff} OR {coastal ecotourism} OR {destructive fishing} OR {local fisheries} OR {artisanal fishers} OR {fisheries rights} OR {species richness} OR {traditional ecological knowledge} OR {small Island development states} OR {marine quota} OR {marine economy} OR {marine policy} ) AND NOT ( {paleoclimate} OR {paleoceanography} OR {radiocarbon} OR {genetics} OR {medicine} OR {drug} OR {engineering} OR {aerosol} ) )
15	TITLE-ABS-KEY ( ( terrestrial OR land OR inland OR freshwater ) AND ( biodivers* OR {species richness} OR bioeconom* OR bio-econom* OR {biological production} OR deforest* OR desertif* OR {earth system} OR {ecological resilience} OR ecosystem* OR eco-system* OR {trophic cascade} OR {trophic level} OR {trophic web} OR {threatened species} OR {endangered species} OR {extinction risk} OR {extinction risks} OR poach* OR {wildlife product} OR {wildlife products} OR {wildlife traffic} OR {wildlife market} OR {wildlife markets} OR {wildlife trafficking} OR {invasive species} OR {alien species} OR {land uses} OR {land use} OR {land uses} OR {land degradation} OR {soil degradation} OR {LULUCF} OR *forest* OR {land conservation} OR wetland* OR mountain* OR dryland* OR {mountainous cover} OR {protected area} OR {protected areas} OR {REDD} OR {forest management} OR {silviculture} OR {timber harvest} OR {illegal logging} OR {slash-and-burn} OR {fire-fallow cultivation} OR {tree cover} OR {soil restoration} OR {land restoration} OR {drought} OR {sustainable land management} OR {mountain vegetation} OR {habitat restoration} OR {Red List species} OR {Red List Index} OR {extinction wave} OR {habitat fragmentation} OR {habitat loss} OR {Nagoya Protocol on Access to Genetic Resources} OR {genetic resources} OR {biological invasion} OR {biodiversity-inclusive} OR {forest stewardship council} OR {rainforest alliance} OR {forest certification} OR {forest auditing} OR {ecotourism} OR {community-based conservation} OR {community based conservation} OR {human-wildlife conflict} ) )
16	TITLE-ABS-KEY ( ( {actual innocence} OR {false confession} OR {armed conflict} OR {armed conflicts} OR {civil conflict} OR {civil conflicts} OR ( war AND ( conflict OR warfare OR democracy OR {Geneva Convention} OR treaty OR peace ) ) OR {peacekeeping} OR ( corruption AND ( {institution} OR {public official} OR {government} OR {bribery} OR {conflict} ) ) OR crime OR crimes OR criminal OR {democratic deficit} OR ( democrati?ation AND ( institutional OR conflict OR decision-making OR society OR politics OR {financial aid} ) ) OR {ethnic conflict} OR {ethnic conflicts} OR exoneration OR genocid* OR homicid* OR murder* OR {human trafficking} OR {criminal justice system} OR {justice system} OR {arbitrary justice} OR refugee* OR terroris* OR violence OR torture OR {effective rule of law} OR {arms flow} OR {transparent institution} OR {transparent institutions} OR {good governance} OR {legal identity for all} OR {freedom of information} OR {human rights institution} OR {human rights activists} OR {fundamental freedom} OR {fundamental freedoms} OR {violent conflict} OR {violent conflicts} OR {peaceful society} OR {effective institution} OR {effective institutions} OR {accountable institution} OR {accountable institutions} OR {inclusive institution} OR {inclusive institutions} OR {child abuse} OR {arbitrary detention} OR {unsentenced detention} OR {judicial system} OR {criminal tribunal} OR {inclusive society} OR {inclusive societies} OR {responsive institution} OR {responsive institutions} OR {fair society} OR {fair societies} OR {legal remedy} OR {legal remedies} OR {independence of judiciary} OR {independent judiciary} OR {separation of powers} OR extremism OR {war crime} OR {peaceful society} OR {organized crime} OR {illicit transfer} OR {illicit money} OR {arms trafficking} OR {cybercrime} OR {insurgence} OR {democratic institution} OR {political instability} OR ( {political decision-making} AND ( responsive OR inclusive OR participatory OR representative ) ) OR {Aarhus Convention} OR {press freedom} OR {freedom of speech} ) AND NOT ( {disease} OR {genetics} ) )
"""

queries_ods = {}

# Processa cada linha do texto bruto
for line in raw_scopus_queries.strip().split('\n'):
    parts = line.split('\t')
    if len(parts) == 2 and parts[0].isdigit():
        ods_num = int(parts[0])
        query_text = parts[1]
        
        # 1. Pega o conteúdo de dentro de "TITLE-ABS-KEY ( ... )"
        # O ".strip()" remove espaços extras no início/fim
        match = re.search(r'TITLE-ABS-KEY \((.*)\)', query_text, re.IGNORECASE)
        if not match:
            continue
        
        # 2. Limpa a string da query
        clean_query = match.group(1).strip()
        
        # Remove a parte do AND NOT no final da query, se existir
        clean_query = re.sub(r'AND NOT .*$', '', clean_query)
        
        # Remove parênteses externos, chaves, wildcards e aspas
        # Converte para minúsculas
        clean_query = clean_query.strip('() ')
        clean_query = clean_query.replace('{', '').replace('}', '')
        clean_query = clean_query.replace('*', '').replace('?', '')
        clean_query = clean_query.replace('“', '').replace('”', '')
        clean_query = clean_query.lower()
        
        # 3. Substitui operadores lógicos por '|'
        # Usamos regex para substituir ' or ' e ' and ' com espaços ao redor
        clean_query = re.sub(r'\s+(or|and)\s+', '|', clean_query)
        
        # Remove todos os parênteses restantes
        clean_query = re.sub(r'[\(\)]', '', clean_query)
        
        # 4. Cria uma lista de termos, remove espaços e junta tudo com '|'
        # Isso garante que não haja itens vazios ou barras duplas (||)
        terms = [term.strip() for term in clean_query.split('|') if term.strip()]
        final_query_string = '|'.join(sorted(list(set(terms)))) # Ordena e remove duplicados
        
        # 5. Adiciona ao dicionário final
        if ods_num in ods_names:
            queries_ods[ods_names[ods_num]] = final_query_string

# Para verificar o resultado, vamos imprimir um dos ODS formatados
print("Dicionário de queries gerado com sucesso!")
print(f"{len(queries_ods)} queries de ODS foram definidas.")
print("\n--- Exemplo para ODS 7 ---")
print(queries_ods["ODS 7: Energia Limpa e Acessível"])

# (Opcional) Para ver o dicionário inteiro de forma legível:
import pprint
pprint.pprint(queries_ods)

Dicionário de queries gerado com sucesso!
16 queries de ODS foram definidas.

--- Exemplo para ODS 7 ---
2000 watt society|affordable electricity|clean cooking fuel|clean energy technology|clean fuel|developing countries|district heat|district heating|domestic energy consumption|electricity|electricity consumption|energiewende|energy access|energy conservation|energy consumption|energy development|energy efficiency|energy equity|energy governance|energy justice|energy ladder|energy policy|energy poverty|energy security|energy transition|foreign development aid|fuel poverty|fuel switching|heat network|hybrid renewable energy system|hybrid renewable energy systems|hydrogen production|life cycle assessment|life cycle assessments|life-cycle assessment|life-cycle assessments|lithium-ion batteries|lithium-ion battery|low-carbon society|official development assistance|photocatalytic water splitting|photochemistry|photovoltaic|reliable electricity|renewable|renewable energy|residential energy 

### Passo 3: Execução da Classificação
Agora, vamos iterar sobre cada ODS, aplicar a busca textual e criar uma nova coluna booleana (True/False) para cada um, indicando se o artigo corresponde aos termos daquele ODS.

In [12]:

#### Célula 9: Código (Loop de Classificação)```python
# Itera sobre o dicionário de queries
for nome_ods, termos in queries_ods.items():
    print(f"Processando {nome_ods}...")
    # Cria uma nova coluna para cada ODS.
    # df['texto_busca'].str.contains() retorna True se encontrar qualquer um dos termos, e False caso contrário.
    df[nome_ods] = df['texto_busca'].str.contains(termos, regex=True, case=False)

print("\nClassificação concluída!")
# Mostra as novas colunas criadas para os 5 primeiros artigos
colunas_ods_criadas = list(queries_ods.keys())
df[['eid'] + colunas_ods_criadas].head() # Use 'eid' ou outro ID do seu artigo

Processando ODS 1: Erradicação da Pobreza...
Processando ODS 2: Fome Zero e Agricultura Sustentável...
Processando ODS 3: Saúde e Bem-Estar...
Processando ODS 4: Educação de Qualidade...
Processando ODS 5: Igualdade de Gênero...
Processando ODS 6: Água Potável e Saneamento...
Processando ODS 7: Energia Limpa e Acessível...
Processando ODS 8: Trabalho Decente e Crescimento Econômico...
Processando ODS 9: Indústria, Inovação e Infraestrutura...
Processando ODS 10: Redução das Desigualdades...
Processando ODS 11: Cidades e Comunidades Sustentáveis...
Processando ODS 12: Consumo e Produção Responsáveis...
Processando ODS 13: Ação Contra a Mudança Global do Clima...
Processando ODS 14: Vida na Água...
Processando ODS 15: Vida Terrestre...
Processando ODS 16: Paz, Justiça e Instituições Eficazes...

Classificação concluída!


,eid,ODS 1: Erradicação da Pobreza,ODS 2: Fome Zero e Agricultura Sustentável,ODS 3: Saúde e Bem-Estar,ODS 4: Educação de Qualidade,ODS 5: Igualdade de Gênero,ODS 6: Água Potável e Saneamento,ODS 7: Energia Limpa e Acessível,ODS 8: Trabalho Decente e Crescimento Econômico,"ODS 9: Indústria, Inovação e Infraestrutura",ODS 10: Redução das Desigualdades,ODS 11: Cidades e Comunidades Sustentáveis,ODS 12: Consumo e Produção Responsáveis,ODS 13: Ação Contra a Mudança Global do Clima,ODS 14: Vida na Água,ODS 15: Vida Terrestre,"ODS 16: Paz, Justiça e Instituições Eficazes"
0,2-s2.0-85162981044,False,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False
1,2-s2.0-85149013333,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False
2,2-s2.0-85160833451,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,2-s2.0-85167577291,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
4,2-s2.0-85143804200,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False


### Passo 4: Consolidação e Salvamento dos Resultados

Para facilitar a análise e a visualização no Looker Studio, vamos criar uma última coluna chamada `ODS_Classificados` que conterá uma lista de todos os ODS aos quais o artigo foi associado.
Finalmente, salvamos o dataframe completo em um novo arquivo CSV.

In [13]:
# Cria uma lista com os nomes das colunas ODS que foram criadas
colunas_ods = list(queries_ods.keys())

# Função para aplicar em cada linha do dataframe
def consolidar_ods(row):
    ods_encontrados = [col for col in colunas_ods if row[col] == True]
    if not ods_encontrados:
        return "Nenhum ODS identificado"
    return ", ".join(ods_encontrados) # Retorna uma string separada por vírgula

# Aplica a função para criar a coluna consolidada
df['ODS_Classificados'] = df.apply(consolidar_ods, axis=1)

print("Coluna 'ODS_Classificados' criada com sucesso.")
print("\nContagem de artigos por classificação de ODS:")
print(df['ODS_Classificados'].value_counts().head(10))

# Salva o dataframe final em um novo arquivo CSV
df.to_csv(NOME_ARQUIVO_SAIDA, index=False, encoding='utf-8-sig')

print(f"\nResultados salvos com sucesso no arquivo: '{NOME_ARQUIVO_SAIDA}'")

Coluna 'ODS_Classificados' criada com sucesso.

Contagem de artigos por classificação de ODS:
ODS_Classificados
Nenhum ODS identificado                                                  222
ODS 15: Vida Terrestre                                                    92
ODS 16: Paz, Justiça e Instituições Eficazes                              44
ODS 15: Vida Terrestre, ODS 16: Paz, Justiça e Instituições Eficazes      26
ODS 6: Água Potável e Saneamento, ODS 15: Vida Terrestre                  21
ODS 11: Cidades e Comunidades Sustentáveis                                17
ODS 6: Água Potável e Saneamento                                          17
ODS 11: Cidades e Comunidades Sustentáveis, ODS 15: Vida Terrestre        16
ODS 14: Vida na Água, ODS 15: Vida Terrestre                              14
ODS 13: Ação Contra a Mudança Global do Clima, ODS 15: Vida Terrestre     12
Name: count, dtype: int64

Resultados salvos com sucesso no arquivo: 'artigos_classificados_ods.csv'
